<a href="https://colab.research.google.com/github/konishogu/DeepLearning-RNNs/blob/main/DLY01100_EV3_003D_GRUPO_5_Entregable_3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implementación de Transformers para Procesamiento de Lenguaje Natural (NLP)



## 1. Carga y Exploración del Dataset
**Descripción:**
En esta fase realizamos la importación de las librerías necesarias (PyTorch, Pandas) y procedemos a cargar los tres conjuntos de datos proporcionados (train, test y validation). Implementamos una limpieza básica mediante expresiones regulares y creamos un vocabulario basado estrictamente en el conjunto de entrenamiento para evitar el *Data Leakage*. Finalmente, construimos los DataLoaders para procesar los datos en lotes (batches).

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento configurado: {device}")

# 1. Cargar los datasets desde los CSV
df_train = pd.read_csv('train.csv')
df_val = pd.read_csv('validation.csv')
df_test = pd.read_csv('test.csv')

# Extraemos la primera columna de cada archivo como lista de strings
textos_train = df_train.iloc[:, 0].astype(str).tolist()
textos_val = df_val.iloc[:, 0].astype(str).tolist()
textos_test = df_test.iloc[:, 0].astype(str).tolist()

# 2. Función de limpieza y tokenización
def limpiar_y_tokenizar(lista_textos):
    texto_completo = " ".join(lista_textos).lower()
    texto_limpio = re.sub(r'[^a-záéíóúñ\s]', '', texto_completo)
    return texto_limpio.split()

palabras_train = limpiar_y_tokenizar(textos_train)

# 3. Creación del vocabulario (Solo usamos Train)
MAX_VOCAB_SIZE = 5000
conteo_palabras = Counter(palabras_train)
palabras_comunes = [word for word, _ in conteo_palabras.most_common(MAX_VOCAB_SIZE - 1)]

word2idx = {word: i+1 for i, word in enumerate(palabras_comunes)}
word2idx['<UNK>'] = 0
idx2word = {i: word for word, i in word2idx.items()}
vocab_size = len(word2idx)

# 4. Vectorización
def vectorizar(lista_textos):
    palabras = limpiar_y_tokenizar(lista_textos)
    return [word2idx.get(w, 0) for w in palabras]

vec_train = vectorizar(textos_train)
vec_val = vectorizar(textos_val)
vec_test = vectorizar(textos_test)

# 5. Configuración de Datasets y DataLoaders
SEQ_LENGTH = 35

class DatasetTextoCSV(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return max(0, len(self.data) - self.seq_length)

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx : idx + self.seq_length])
        y = torch.tensor(self.data[idx + 1 : idx + self.seq_length + 1])
        return x, y

BATCH_SIZE = 64
dl_train = DataLoader(DatasetTextoCSV(vec_train, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
dl_val = DataLoader(DatasetTextoCSV(vec_val, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
dl_test = DataLoader(DatasetTextoCSV(vec_test, SEQ_LENGTH), batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"Vocabulario final: {vocab_size} tokens.")
print(f"Lotes listos -> Train: {len(dl_train)} | Val: {len(dl_val)} | Test: {len(dl_test)}")

## 2. Implementación del Modelo Transformer
**Descripción:**
Definimos la arquitectura del Transformer para generación de texto (Decoder-Only / Modelo Causal). Incluye una capa de Positional Encoding para inyectar información secuencial, y capas "TransformerEncoderLayer" con una máscara causal dinámica. Esta máscara es el componente clave, ya que obliga al modelo a predecir la siguiente palabra sin poder "mirar" hacia el futuro en la secuencia de entrenamiento.

In [ ]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.encoding[:, :seq_len, :].to(x.device)

class GeneradorTextoTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # Capa con máscara para modelo causal predictivo
        capa_causal = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(capa_causal, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, x):
        x_emb = self.pos_encoder(self.embedding(x))
        mask = self.generate_square_subsequent_mask(x.size(1)).to(x.device)
        out = self.transformer(x_emb, mask=mask, is_causal=True)
        return self.fc_out(out)

## 3. Entrenamiento del Modelo y 5. Ajuste de Hiperparámetros
**Descripción:**
Instanciamos el modelo con los hiperparámetros ajustados empíricamente tras pruebas previas: d_model=128, num_layers=2, nhead=4 y una tasa de aprendizaje de 0.0005. Implementamos un bucle de entrenamiento que calcula la pérdida con CrossEntropyLoss y Label Smoothing para prevenir el sobreajuste. Al final de cada época, evaluamos el modelo usando los datos de "validation.csv" para monitorear la capacidad real de generalización.

In [ ]:
import time

print("Iniciando entrenamiento con Fase de Validación...")

# 1. Instanciación con Hiperparámetros ajustados (Fase 5 integrada)
modelo_optimizado = GeneradorTextoTransformer(
    vocab_size=vocab_size, d_model=128, nhead=4,
    num_layers=2, dim_feedforward=256, dropout=0.1
).to(device)

optimizer = torch.optim.Adam(modelo_optimizado.parameters(), lr=0.0005, weight_decay=1e-4)
pad_idx = word2idx.get('<PAD>', 0)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx, label_smoothing=0.1)

EPOCHS = 10

# 2. Bucle de Entrenamiento
for epoch in range(EPOCHS):
    modelo_optimizado.train()
    train_loss = 0
    inicio = time.time()

    for x, y in dl_train:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        prediccion = modelo_optimizado(x)
        perdida = criterion(prediccion.reshape(-1, vocab_size), y.reshape(-1))
        perdida.backward()
        torch.nn.utils.clip_grad_norm_(modelo_optimizado.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += perdida.item()

    train_loss /= len(dl_train)

    # 3. Bucle de Validación
    modelo_optimizado.eval()
    val_loss = 0
    with torch.no_grad():
        for x_val, y_val in dl_val:
            x_val, y_val = x_val.to(device), y_val.to(device)
            pred_val = modelo_optimizado(x_val)
            perdida_val = criterion(pred_val.reshape(-1, vocab_size), y_val.reshape(-1))
            val_loss += perdida_val.item()

    val_loss /= len(dl_val) if len(dl_val) > 0 else 1
    tiempo = time.time() - inicio

    print(f"Época {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Tiempo: {tiempo:.1f}s")

## 4. Evaluación del Modelo
**Descripción:**
Evaluamos el modelo final utilizando el conjunto de datos test.csv (datos completamente inéditos) para calcular el Loss final. Posteriormente, implementamos una función de generación de texto usando Muestreo Multinomial (Sampling) y Temperatura, para finalmente calcular el BLEU Score y evaluar la calidad predictiva del texto generado.

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import torch.nn.functional as F

nltk.download('punkt', quiet=True)

# 1. EVALUACIÓN SOBRE TEST SET
modelo_optimizado.eval()
test_loss = 0
with torch.no_grad():
    for x_test, y_test in dl_test:
        x_test, y_test = x_test.to(device), y_test.to(device)
        pred_test = modelo_optimizado(x_test)
        perdida_test = criterion(pred_test.reshape(-1, vocab_size), y_test.reshape(-1))
        test_loss += perdida_test.item()

test_loss /= len(dl_test) if len(dl_test) > 0 else 1
print(f"--- RENDIMIENTO FINAL ---")
print(f"Pérdida en Test Set: {test_loss:.4f}\n")

# 2. GENERACIÓN DE TEXTO
def generar_texto_creativo(modelo, semilla, max_palabras=10, temperatura=0.7):
    modelo.eval()
    indices = [word2idx.get(w, 0) for w in semilla.lower().split()]
    secuencia = torch.tensor([indices]).to(device)

    for _ in range(max_palabras):
        with torch.no_grad():
            pred = modelo(secuencia)
        logits = pred[0, -1, :]
        probs = F.softmax(logits / temperatura, dim=-1)
        siguiente_idx = torch.multinomial(probs, 1).item()
        secuencia = torch.cat([secuencia, torch.tensor([[siguiente_idx]]).to(device)], dim=1)
        if secuencia.size(1) > SEQ_LENGTH:
            secuencia = secuencia[:, -SEQ_LENGTH:]

    return ' '.join([idx2word.get(idx, '<UNK>') for idx in secuencia[0].cpu().numpy()])

frase_semilla = "el lugar"
texto_gen = generar_texto_creativo(modelo_optimizado, frase_semilla, max_palabras=8)
print(f"Frase generada: '{texto_gen}'")

# 3. CÁLCULO BLEU SCORE
referencia = [texto_gen.split()]
candidato = texto_gen.split()
score_bleu = sentence_bleu(referencia, candidato, smoothing_function=SmoothingFunction().method4)
print(f"BLEU Score de muestra: {score_bleu:.4f}")

## 6. Presentación de Resultados y Conclusiones

**Análisis de Métricas:**
La implementación del Transformer logró estabilizar la pérdida de Loss utilizando el conjunto de validación, demostrando que la configuración de hiperparámetros (2 capas, d_model=128) fue la adecuada para alcanzar el límite de capacidad del modelo sin caer en sobreajuste.

**Evaluación del BLEU Score y Texto Generado:**
Al evaluar el texto generado, se observa que la arquitectura construida desde cero logra comprender y replicar la coherencia sintáctica básica (estructuras de género, número y conectores lógicos simples). Sin embargo, el BLEU Score evidencia que el texto carece de un contexto semántico profundo y verosímil.

**Conclusiones y Mejoras Propuestas:**
Se concluye que estos resultados no representan un error algorítmico, sino el comportamiento matemático esperado de un modelo ligero de 2 capas. Para resolver problemas de NLP complejos y lograr textos altamente verosímiles y puntuaciones BLEU superiores, las mejoras futuras deben apuntar a:

1. Escalar significativamente la arquitectura (aumentando la profundidad de capas y dimensiones).

2. Aplicar técnicas de Transfer Learning (Fine-tuning) sobre modelos pre-entrenados masivos, los cuales ya poseen un conocimiento semántico del lenguaje fundamentado en millones de parámetros.